# Building a Document Loader for RAG (Retrieval-Augmented Generation)

Welcome to this beginner-friendly tutorial! In this notebook, we are going to explore how to load data into a format that AI models can understand using **LangChain**. Loading documents is the very first step in building any RAG system.

We will explore different ways to load data, covering both simple **Text files** and **PDF files**:

### 1. The `TextLoader` Approach (Manual)
The `TextLoader` is designed to read a **single file at a time**. Because we have multiple text files in our `data/text_files/` directory, we had to write a Python `for` loop to manually find every file, open it with `TextLoader`, and add it to our list. While this approach gives you a lot of control (for example, if you want to skip certain files), it can be tedious to write out the loop.

### 2. The `DirectoryLoader` Approach (Automatic - Text Files)
The `DirectoryLoader` is a much more powerful tool. Instead of writing a loop, you just point `DirectoryLoader` at an **entire folder**. Under the hood, it automatically finds all the files that match your criteria (like all `.txt` files) and uses the `TextLoader` to read them all for you. This approach is much cleaner and is highly recommended when dealing with many files!

### 3. Loading PDFs with `DirectoryLoader`
Text files are simple, but what about PDFs? LangChain makes this incredibly easy! By using the exact same `DirectoryLoader` but changing its internal reader to `PyPDFLoader`, we can automatically scan a folder and extract the text from every single page of every PDF inside it.

Let's dive into the code and see all three methods in action below!

In [1]:
sample_texts={
    "../data/text_files/space_exploration.txt":"""Space Exploration Basics

Space exploration is the use of astronomy and space technology to explore outer space. Physical exploration of space is conducted both by human spaceflights and by robotic spacecraft. Currently, humanity is focused on returning to the Moon and eventually reaching Mars.

The history of space exploration began with the launch of Sputnik 1 by the Soviet Union in 1957. This triggered the Space Race, culminating in the Apollo 11 mission in 1969, which successfully landed the first humans on the Moon.

In recent years, the commercialization of space has accelerated, with private companies pioneering reusable launch systems, lowering the cost of access to space. Robotic missions like the James Webb Space Telescope continue to push the boundaries of human knowledge by observing distant galaxies, exoplanets, and remnants of the early universe.""",
    
    "../data/text_files/quantum_computing.txt": """Quantum Computing Fundamentals

Quantum computing is a rapidly-emerging technology that harnesses the laws of quantum mechanics to solve problems too complex for classical computers. It utilizes quantum bits, or qubits, allowing for superposition and entanglement, drastically speeding up certain calculations.

Unlike classical bits that must be either a 0 or a 1, a qubit can exist in a superposition of both states simultaneously. Combined with quantum entanglement, this allows quantum computers to process multiple possibilities at an unprecedented scale.

Major technology companies and research institutions are actively developing quantum hardware. These advancements hold the potential to revolutionize fields such as cryptography, drug discovery, materials science, and complex system optimization."""

}

import os
for filepath,content in sample_texts.items():
    # Adding a safety check to automatically make the data directories if missing!
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath,'w',encoding="utf-8") as f:
        f.write(content)

print("✅ Sample text files created!")

✅ Sample text files created!


In [2]:
# Import the TextLoader from Langchain's community module.
# TextLoader is specifically designed to read plain text files into Langchain Document formats.
from langchain_community.document_loaders import TextLoader


# Define the folder where our text files are saved
text_folder = "../data/text_files/"

# Create an empty list to store all our loaded LangChain documents
all_documents = []

print("Starting to load documents using LangChain...\n")

# Loop through every file in the directory
for filename in os.listdir(text_folder):
    # Check if the file is a standard text file
    if filename.endswith(".txt"):
        # Create the full relative path to the file
        file_path = os.path.join(text_folder, filename)
        
        # Initialize the TextLoader with our specific file path
        # We specify utf-8 encoding to avoid any character reading errors
        loader = TextLoader(file_path, encoding='utf-8')
        
        # .load() reads the file and returns a list containing a LangChain 'Document'
        # We use [0] because .load() always returns a list, even for a single file
        document = loader.load()[0]
        
        # Add this document to our master list of documents
        all_documents.append(document)
        print(f"✅ Successfully loaded: {filename}")

# Let's verify how many documents we loaded in total!
print(f"\nTotal documents loaded into memory: {len(all_documents)}")

# A LangChain 'Document' object has two main components:
# 1. page_content: The actual text string inside the file
# 2. metadata: A dictionary containing information about the file (like its source path)

# Let's print a quick preview of our documents to verify everything works!
for i, doc in enumerate(all_documents):
    print(f"\n--- Preview of Document {i+1} ---")
    print(f"Source Path: {doc.metadata['source']}")
    print(f"Text Preview: {doc.page_content[:150]}...")  # Prints the first 150 characters


Starting to load documents using LangChain...

✅ Successfully loaded: space_exploration.txt
✅ Successfully loaded: quantum_computing.txt

Total documents loaded into memory: 2

--- Preview of Document 1 ---
Source Path: ../data/text_files/space_exploration.txt
Text Preview: Space Exploration Basics

Space exploration is the use of astronomy and space technology to explore outer space. Physical exploration of space is cond...

--- Preview of Document 2 ---
Source Path: ../data/text_files/quantum_computing.txt
Text Preview: Quantum Computing Fundamentals

Quantum computing is a rapidly-emerging technology that harnesses the laws of quantum mechanics to solve problems too ...


In [3]:
# Import the DirectoryLoader and TextLoader from the Langchain community module.
# DirectoryLoader is perfect for reading ENTIRE folders of files at once!
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# Define the folder where our text files are saved
text_folder = "../data/text_files/"

print("Starting to load documents using DirectoryLoader...\n")

# Initialize the DirectoryLoader.
# We pass three important arguments:
# 1. The path to our folder (text_folder)
# 2. 'glob' parameter: "**/*.txt" tells the loader to ONLY look for text files 
#    and ignore everything else (like hidden system files).
# 3. 'loader_cls': We tell the directory loader to use TextLoader under the hood
#    to actually read each text file it finds correctly.
dir_loader = DirectoryLoader(
    text_folder, 
    glob="**/*.txt", 
    loader_cls=TextLoader
)

# The .load() method automatically scans the directory, finds the text files, 
# reads them using TextLoader, and combines them into a single list of Documents!
all_directory_documents = dir_loader.load()

# Let's verify how many documents the DirectoryLoader found!
print(f"✅ DirectoryLoader successfully loaded {len(all_directory_documents)} documents!\n")

# Let's print a quick preview of our documents to verify it worked.
for i, doc in enumerate(all_directory_documents):
    print(f"--- Directory Document {i+1} ---")
    print(f"Source Path: {doc.metadata['source']}")
    print(f"Text Preview: {doc.page_content[:150]}...\n")


Starting to load documents using DirectoryLoader...

✅ DirectoryLoader successfully loaded 2 documents!

--- Directory Document 1 ---
Source Path: ../data/text_files/space_exploration.txt
Text Preview: Space Exploration Basics

Space exploration is the use of astronomy and space technology to explore outer space. Physical exploration of space is cond...

--- Directory Document 2 ---
Source Path: ../data/text_files/quantum_computing.txt
Text Preview: Quantum Computing Fundamentals

Quantum computing is a rapidly-emerging technology that harnesses the laws of quantum mechanics to solve problems too ...



In [4]:
# Now let's learn how to load PDF files!
# We will use the same DirectoryLoader, but this time we pair it with PyPDFLoader
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

# Define the folder where our PDF files are saved
pdf_folder = "../data/pdf/"

print("Starting to load PDF documents...\n")

# Initialize the DirectoryLoader for PDFs.
# 1. We point it to the pdf_folder
# 2. glob="**/*.pdf" ensures it ONLY looks for PDF files
# 3. loader_cls=PyPDFLoader tells it to use the PDF reader for each file it finds
pdf_loader = DirectoryLoader(
    pdf_folder,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
)

# .load() will automatically find and read all PDFs in the folder!
# (Note: PyPDFLoader typically loads each page of a PDF as a separate Document object)
all_pdf_documents = pdf_loader.load()

print(f"✅ Successfully loaded {len(all_pdf_documents)} pages from the PDF files!\n")

# Let's print a quick preview of the first PDF page
if all_pdf_documents:
    first_page = all_pdf_documents[0]
    print("--- Quick Preview of First PDF Page ---")
    print(f"Source Document: {first_page.metadata['source']}")
    # PDFs are typically loaded page-by-page, so metadata also stores the page number
    print(f"Page Number: {first_page.metadata.get('page', 0)}")
    print(f"Text Preview: {first_page.page_content[:150]}...\n")
else:
    print("No PDFs found yet! Drop some PDFs into data/pdf/ and run this again.")


Starting to load PDF documents...

✅ Successfully loaded 120 pages from the PDF files!

--- Quick Preview of First PDF Page ---
Source Document: ../data/pdf/Retrieval-Augmented_Generation_RAG.pdf
Page Number: 0
Text Preview: CATCHWORD
Retrieval-Augmented Generation (RAG)
Michael Klesel • H. Felix Wittmann
Received: 22 July 2024 / Accepted: 7 April 2025 / Published online: ...



In [7]:
all_pdf_documents[0].page_content

'CATCHWORD\nRetrieval-Augmented Generation (RAG)\nMichael Klesel • H. Felix Wittmann\nReceived: 22 July 2024 / Accepted: 7 April 2025 / Published online: 1 June 2025\n/C211The Author(s) 2025\nKeywords Retrieval-augmented generation /C1Artiﬁcial\nintelligence /C1Large language models /C1Information\nretrieval\n1 Introduction\nThe necessity for information is a fundamental aspect of\nhuman nature, and as such, there are ongoing efforts to\nenhance information retrieval with information systems\n(Alavi and Leidner 2001; Alavi et al. 2024). Companies\nare particularly affected by this, as they have extensive data\nat their disposal, and employees need to access it. Unfor-\ntunately, current systems are not able to adequately meet\nemployees’ expectations. In fact, studies have shown that\n79% of employees are dissatisﬁed with the user interfaces\nof enterprise search systems (Cleverley and Burnett 2019).\nThis has led to a need for new approaches that can better\naddress the information ne